# LMCBert reproduction — full split on Colab (v3: graph-centrality summariser)

Paper: Liu et al., *LMCBert*, IEEE T-Cybernetics 2025. Full AAPR split (31.7k train / 1k val / 1k test), zero-cost pipeline.

**Before running:** Runtime → Change runtime type → **T4 GPU**. Then run the cells top to bottom.
Everything (prepared data, checkpoints, results) is saved to your Google Drive under `MyDrive/lmcbert_colab/`,
so if Colab disconnects you just re-run the notebook and it carries on from the last finished epoch.

One-time setup on Drive: put `lmcbert_code.zip` in `MyDrive/lmcbert_colab/`.

In [ ]:
#@title 1. Mount Drive, check GPU
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess
ROOT = '/content/drive/MyDrive/lmcbert_colab'
os.makedirs(ROOT, exist_ok=True)
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], capture_output=True, text=True).stdout or 'NO GPU - change runtime type to T4')
assert os.path.exists(f'{ROOT}/lmcbert_code.zip'), 'Put lmcbert_code.zip in MyDrive/lmcbert_colab/ first'

In [ ]:
#@title 2. Install deps, unpack code, link runs/ to Drive
%pip -q install ijson gdown sentence-transformers
import shutil, os
if os.path.exists('/content/lmcbert_repro'): shutil.rmtree('/content/lmcbert_repro')
os.makedirs('/content/lmcbert_repro'); os.chdir('/content/lmcbert_repro')
!unzip -q -o "$ROOT/lmcbert_code.zip" -d /content/lmcbert_repro
os.makedirs(f'{ROOT}/runs_full', exist_ok=True)
if os.path.islink('runs') or os.path.exists('runs'): os.remove('runs') if os.path.islink('runs') else shutil.rmtree('runs')
os.symlink(f'{ROOT}/runs_full', 'runs')
!ls
import torch, transformers; print('torch', torch.__version__, '| transformers', transformers.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
#@title 3. Data: download AAPR, clean, split 31.7k/1k/1k, summarise, novelty features (once; cached on Drive, ~45 min CPU)
DATA = f'{ROOT}/data/splits'
os.makedirs(f'{ROOT}/data', exist_ok=True)
done = all(os.path.exists(f'{DATA}/{s}.sum.jsonl') for s in ('train','val','test'))
if done:
    print('prepared splits already on Drive - skipping')
else:
    if not os.path.exists('/content/AAPR_Dataset/data1'):
        # public link from github.com/lancopku/AAPR (Yang et al. 2018), 764 MB
        !gdown 1ttKdXUjaFi3eS6zeHITuZjsTGrh6M4ij -O /content/AAPR_Dataset.zip
        !unzip -q -o /content/AAPR_Dataset.zip -d /content/
        !find /content -maxdepth 3 -name data1
    import glob
    raw = os.path.dirname(glob.glob('/content/**/data1', recursive=True)[0])
    os.environ['AAPR_RAW'] = raw
    !python prepare_data.py --out /content/aapr_clean.jsonl
    !python make_splits.py --inp /content/aapr_clean.jsonl --out /content/splits --val 1000 --test 1000
    for s in ('train','val','test'):
        !python summarize.py --inp /content/splits/{s}.jsonl --out /content/splits/{s}.sum.jsonl --backend extractive
    !python enhance/novelty.py --data /content/splits
    shutil.copytree('/content/splits', DATA, dirs_exist_ok=True)
    print('saved to', DATA)
!ls -la "$DATA"
!wc -l "$DATA"/*.sum.jsonl

In [ ]:
#@title 3b. Graph-centrality summaries (Abdalgader & Al Shibli 2021 adapted to sentences; once, cached on Drive, ~1-1.5 h on T4)
SIM = "embed"           #@param ["embed", "tfidf", "wordnet"]   # wordnet = the paper's exact measure but ~6 s/paper -> subset only
CENTRALITY = "indegree" #@param ["indegree", "pagerank"]
GDATA = f'{ROOT}/data/splits_graph_{SIM}_{CENTRALITY}'
os.makedirs(GDATA, exist_ok=True)
if all(os.path.exists(f'{GDATA}/{s}.sum.jsonl') for s in ('train','val','test')):
    print('graph summaries already on Drive - skipping')
else:
    for s in ('val','test','train'):   # small ones first so a failure shows early
        if not os.path.exists(f'{GDATA}/{s}.sum.jsonl'):
            !python -u summarize.py --inp "$DATA/{s}.jsonl" --out "$GDATA/{s}.sum.jsonl.part" --backend graph --sim $SIM --centrality $CENTRALITY --resume
            !mv "$GDATA/{s}.sum.jsonl.part" "$GDATA/{s}.sum.jsonl"
    !python enhance/novelty.py --data "$GDATA"
!wc -l "$GDATA"/*.sum.jsonl
import json
r = json.loads(open(f"{GDATA}/val.sum.jsonl").readline()); print(r["summary_backend"]); print(r["summary"][:600])

In [ ]:
#@title 4. (optional, CPU ~20 min) TF-IDF sanity baselines
RUN_CLASSIC = False  #@param {type:"boolean"}
if RUN_CLASSIC:
    !python baselines_classic.py --data "$DATA" 

In [ ]:
#@title 5. Training runs (resumable; re-run this cell after any disconnect)
MAX_LEN = 512      #@param [256, 384, 512] {type:"raw"}   # paper: 512. 256 is ~2x faster.
EPOCHS = 10        #@param {type:"integer"}                # paper: 80 (with early stopping 10 is enough to rank configs)
PATIENCE = 3       #@param {type:"integer"}
BATCH, ACCUM = (16, 2) if MAX_LEN >= 512 else (32, 1)     # effective batch 32 as in the paper
COMMON = f'--data "{DATA}" --epochs {EPOCHS} --patience {PATIENCE} --batch {BATCH} --accum {ACCUM} --max_len {MAX_LEN} --lr 2e-5 --fp16 --resume --skip_done --clean --progress 200'

RUNS = [  # (name, extra args) - in priority order; each ~2-5 h on a T4 at 512, less at 256
    ('bert_abstract',           '--input abstract --contrastive none'),                        # Table II baseline (paper 63.4)
    ('lmcbert_lam0.1',          '--input summary --contrastive moco --lam 0.1 --queue 4096'),   # best lambda on the subset
    ('lmcbert_lam0.5',          '--input summary --contrastive moco --lam 0.5 --queue 4096'),   # paper-style lambda (paper 68.0)
    ('enh_novelty',             '--input summary --contrastive moco --lam 0.1 --queue 4096 --features novelty'),  # Enhancement 1
    ('ablation_wo_contrastive', '--input summary --contrastive none'),                         # Table III
    ('ablation_wo_queue',       '--input summary --contrastive inbatch --lam 0.5'),            # Table III
    ('ablation_wo_llm',         '--input sections --contrastive moco --lam 0.5 --queue 4096'), # Table III
    ('enh_multimodal',          '--input summary_captions --contrastive moco --lam 0.1 --queue 4096 --features struct'),  # Enhancement 2
]
for name, extra in RUNS:
    print(f'\n=================== {name} ===================', flush=True)
    !python -u train.py $COMMON $extra --name $name

# ---- same two best recipes, but fed the graph-centrality summaries (Enhancement 4)
GCOMMON = COMMON.replace(f'--data "{DATA}"', f'--data "{GDATA}"')
GRUNS = [
    (f'graph_{SIM}_{CENTRALITY}_bert_summary',   '--input summary --contrastive none'),
    (f'graph_{SIM}_{CENTRALITY}_lmcbert_lam0.1', '--input summary --contrastive moco --lam 0.1 --queue 4096'),
]
for name, extra in GRUNS:
    print(f'\n=================== {name} ===================', flush=True)
    !python -u train.py $GCOMMON $extra --name $name
!python collect_results.py

In [ ]:
#@title 6. Results so far
!python collect_results.py
!cp runs/results.md runs/results.csv "$ROOT/"
print('results.md / results.csv copied to', ROOT)